In [1]:
#############
## IMPORTS ##
#############
import os
import torch
from torchvision.datasets import OxfordIIITPet
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

######################
## DEVICE SELECTION ##
######################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## ResNet-18 From Scratch (Classification)

In [2]:


############################
## IMPORT DATA (CIFAR-10) ##
############################
# Transforms for data augmentation and normalization for the training data.
# RandomCrop with padding of 4 and RandomHorizontalFlip are common augmentations for CIFAR-10. These help improve generalization by providing more varied training examples. The normalization values are the mean and stddev of CIFAR-10 dataset.
train_tfms = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
# For the test set, we only normalize the images without any augmentation.
test_tfms = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

# Import CIFAR-10 dataset with the defined transforms. Convert the datasets into DataLoader for batching. Reduce the training set to a subset of N samples for faster experimentation.
train_set = torchvision.datasets.CIFAR10("./data", train=True,          
                                            download=True, transform=train_tfms)
test_set  = torchvision.datasets.CIFAR10("./data", train=False,         
                                         download=True, 
                                         transform=test_tfms)
N = 5000
subset_indices = list(range(N))
train_set = Subset(train_set, subset_indices)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

/Users/butlerju/Library/Python/3.13/lib/python/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
#########################################
## BASE FUNCTIONS AND CLASS FOR RESNET ##
#########################################
def conv3x3(in_planes, out_planes, stride=1):
    """
    Inputs:
        in_planes : Number of input channels
        out_planes: Number of output channels
        stride    : Stride for the convolution
    Outputs:
        3x3 convolution with padding.
    The purpose of this function is to create a 3x3 convolutional layer with specified input and output channels, stride, and padding of 1 to maintain spatial dimensions. In a ResNet architecture, 3x3 convolutions are commonly used for feature extraction while preserving the spatial resolution of the input feature maps.
    """
    # 3x3 convolution with padding
    return nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """
    Inputs:
        in_planes : Number of input channels
        out_planes: Number of output channels
        stride    : Stride for the convolution
    Outputs:
        1x1 convolution.
    The purpose of this function is to create a 1x1 convolutional layer with specified input and output channels and stride. In a ResNet architecture, 1x1 convolutions are often used for dimensionality reduction or expansion, allowing the network to adjust the number of feature channels without affecting the spatial dimensions of the feature maps.
    """
    return nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    """
    The BasicBlock class defines a basic building block for ResNet architectures. It consists of two 3x3 convolutional layers, each followed by batch normalization and ReLU activation. The block also includes a skip connection that adds the input to the output of the second convolutional layer, allowing for better gradient flow during training. If the input and output dimensions differ, a downsample layer is applied to the input to match dimensions before addition.
    """
    # Expansion factor for the number of output channels. This is 1 for BasicBlock. Its function is to define how much the number of output channels increases compared to the number of input channels. In BasicBlock, the number of output channels remains the same as the number of input channels, hence the expansion factor is 1.
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        """
        Inputs:
            in_planes : Number of input channels
            planes    : Number of output channels for the convolutions
            stride    : Stride for the first convolution
            downsample: Downsampling layer to match dimensions if needed
        Outputs:
            None    
        Initializes the BasicBlock with two convolutional layers, batch normalization, ReLU activation, and an optional downsampling layer.
        """
        super().__init__()
        # Convolutional layer 1, batch normalization, and ReLU activation
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        # Convolutional layer 2 and batch normalization
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        # Downsample layer to match dimensions if needed.
        self.downsample = downsample

    def forward(self, x):
        """
        Inputs:
            x : Input tensor
        Outputs:
            out : Output tensor after passing through the BasicBlock
        Performs the forward pass of the BasicBlock, applying two convolutional layers with batch normalization and ReLU activation, and adding the input (possibly downsampled) to the output of the second convolutional layer.
        """
        # Save the input tensor for the skip connection.
        identity = x
        # First convolutional layer with batch normalization and ReLU activation
        out = self.relu(self.bn1(self.conv1(x)))
        # Second convolutional layer with batch normalization
        out = self.bn2(self.conv2(out))
        # If downsampling is needed, apply it to the input.
        if self.downsample is not None:
            identity = self.downsample(x)
        # Add the skip connection and apply ReLU activation
        out = self.relu(out + identity)
        return out

In [ ]:
class ResNet(nn.Module):
    """"
    Constructs a ResNet model using the specified block type and layer configuration.
    Inputs:
        block: The type of block to use (e.g., BasicBlock)
        layers: List specifying the number of blocks in each layer
        num_classes: Number of output classes for classification
        width: Base width (number of channels) for the first layer
    Outputs:
        None
    """
    def __init__(self, block, layers, num_classes=10, width=64):
        """
            Inputs:
                block      : The type of block to use (e.g., BasicBlock)
                layers     : List specifying the number of blocks in each layer
                num_classes: Number of output classes for classification
                width      : Base width (number of channels) for the first layer
            Outputs:
                None
            Initializes the ResNet model with the specified architecture.
        """
        super().__init__()
        # Initial number of input channels
        self.in_planes = width
        # Initial convolutional layer, batch normalization, and ReLU activation
        self.conv1 = conv3x3(3, width, stride=1)  # CIFAR: 3x3, stride 1
        self.bn1 = nn.BatchNorm2d(width)
        self.relu = nn.ReLU(inplace=True)
        # Create the four layers of the ResNet. _make_layer constructs each layer with the specified number of blocks and channels. It is defined below.
        self.layer1 = self._make_layer(block, width,   layers[0], stride=1)  # 32x32
        self.layer2 = self._make_layer(block, width*2, layers[1], stride=2)  # 16x16
        self.layer3 = self._make_layer(block, width*4, layers[2], stride=2)  # 8x8
        self.layer4 = self._make_layer(block, width*8, layers[3], stride=2)  # 4x4
        # Average pooling layer and fully connected layer for classification.
        self.avg = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(width*8*block.expansion, num_classes)
        # Initialize weights
        self._init_weights()

    def _make_layer(self, block, planes, blocks, stride):
        """
        Inputs:
            block : The type of block to use (e.g., BasicBlock)
            planes: Number of output channels for the blocks in this layer
            blocks: Number of blocks to create in this layer
            stride: Stride for the first block in this layer
        Outputs:
            nn.Sequential containing the blocks for this layer.
        Constructs a layer of the ResNet consisting of multiple blocks.
        """
        # Set up downsampling to None.
        downsample = None
        # If the stride is not 1 or the input and output channels differ, create a downsampling layer.
        if stride != 1 or self.in_planes != planes*block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.in_planes, planes*block.expansion, stride),
                nn.BatchNorm2d(planes*block.expansion),
            )
        # Create the list of blocks for this layer. Currenly only the first block may have a different stride and downsampling.
        layers = [block(self.in_planes, planes, stride, downsample)]
        # Update the number of input channels for the next blocks.
        self.in_planes = planes*block.expansion
        # Add the remaining blocks with stride 1.
        for _ in range(1, blocks):
            layers.append(block(self.in_planes, planes))
        # Convert the list of blocks into a Sequential module and return it.
        return nn.Sequential(*layers)

    def _init_weights(self):
        """
        Inputs:
            None
        Initializes the weights of the model using Kaiming normal initialization for convolutional layers, and normal initialization for linear layers. Kaiming initialization helps maintain the variance of activations through the layers, which is particularly important for deep networks like ResNet.
        """
        # Initialize weights for convolutional, batch normalization, and linear layers. If the layer is a Conv2d, Kaiming normal initialization is applied to its weights. For BatchNorm2d layers, weights are initialized to 1 and biases to 0. For Linear layers, weights are initialized with a normal distribution (mean 0, stddev 0.01) and biases to 0.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        Inputs:
            x : Input tensor
        Outputs:
            x : Output tensor after passing through the ResNet model
        Forward pass through the ResNet model.
        """
        # Initial convolutional layer with batch normalization and ReLU activation
        x = self.relu(self.bn1(self.conv1(x)))
        # Pass through the four ResNet layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        # Average pooling and fully connected layer for classification
        x = self.avg(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        # Return the output
        return x



In [5]:
def resnet18(num_classes=10):
    """"
    Inputs:
        num_classes: Number of output classes for classification
    Outputs:        
        ResNet model instance
    Constructs a ResNet-18 model.
    """
    # Use the ResNet class with BasicBlock and [2,2,2,2] layers to create ResNet-18. The [2,2,2,2] means that there are 2 blocks in each of the 4 layers.
    return ResNet(BasicBlock, [2,2,2,2], num_classes=num_classes)

# Initialize the ResNet-18 model and move it to the selected device (CPU or GPU).
model = resnet18(10).to(device)



In [6]:
##############
## Training ##
##############
# Define the loss function and optimizer for training the model. CrossEntropyLoss is commonly used for multi-class classification problems, and Adam is an adaptive learning rate optimization algorithm that is widely used for training deep learning models.
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train=True):
    """
    Inputs:
        loader: DataLoader for the dataset
        train : Boolean indicating whether to train or evaluate
    Outputs:
        loss_avg: Average loss over the epoch
        acc      : Accuracy over the epoch
    Create a function to run one epoch of training or evaluation.
    """
    # If the mode is training, set the model to training mode; otherwise, set it to evaluation mode.
    model.train() if train else model.eval()

    # Initialize counters for total samples, correct predictions, and cumulative loss.
    total = 0
    correct = 0
    loss_sum =  0.0
    # Iterate over the data in the DataLoader, one batch at a time.
    for x, y in loader:
        # Move the input data and labels to the selected device (CPU or GPU).
        x = x.to(device)
        y = y.to(device)
        # If in training mode, zero the gradients of the optimizer.
        if train: optimizer.zero_grad(set_to_none=True)
        # Forward pass through the model to get logits (outouts)
        logits = model(x)
        # Compute the loss between the logits and the true labels.
        loss = loss_func(logits, y)
        # Backpropagation and optimization step if in training mode.
        if train:
            loss.backward()
            optimizer.step()
        # Update the cumulative loss and accuracy counters.
        loss_sum += loss.item() * x.size(0)
        # Update the number of correct predictions.
        correct += (logits.argmax(1) == y).sum().item()
        # Update the total number of samples processed.
        total += x.size(0)
    # Return the average loss and accuracy for the epoch.
    return loss_sum/total, 100.0*correct/total

# Train the model for a specified number of epochs, evaluating on the validation set after each epoch. Save the model if it achieves the best validation accuracy so far. Define the current best accuracy as 0.0.
best_acc = 0.0
# Train for 10 epochs. Change this as needed.
for epoch in range(1, 11):
    print("Epoch:", epoch)
    # Train and evaluate for one epoch. We use the training data to train
    # the model and the test data to evaluate its performance.
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(test_loader,  train=False)
    # If the validation accuracy is better than the best so far, save the model.
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({"model": model.state_dict(), "acc": best_acc, "epoch": epoch}, "resnet18_cifar10_best.pt")

# Print the best validation accuracy and the path where the best model is saved.
print(f"\nBest Val Acc: {best_acc:.2f}%")
print("Saved to:", os.path.abspath("resnet18_cifar10_best.pt"))

# This takes about 20 minutes on my Mac. 

Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10

Best Val Acc: 57.65%
Saved to: /Users/butlerju/DSC399/CourseNotes/DSC399-Spring2026/resnet18_cifar10_best.pt


## U-Net

In [7]:
##########################
## U-NET BUILDING BLOCK ##
##########################
class DoubleConv(nn.Module):
    """
    DoubleConv implements a sequence of two convolutional layers, each followed by a ReLU activation. This block is commonly used in U-Net architectures for image segmentation tasks. The purpose of this block is to extract features from the input while maintaining the spatial dimensions through padding.
    """
    def __init__(self, in_c, out_c):
        """
        Inputs:
            in_c : Number of input channels
            out_c: Number of output channels
        Outputs:
            None
        Initializes the DoubleConv block with two convolutional layers and ReLU activations.
        """
        super().__init__()
        # Create a sequential container with two convolutional layers and ReLU activations. This will be the building blocks of a U-Net architecture.
        self.conv = nn.Sequential(
            # First convolutional layer
            nn.Conv2d(in_c, out_c, 3, padding=1),
            # ReLU activation
            nn.ReLU(inplace=True),
            # Second convolutional layer
            nn.Conv2d(out_c, out_c, 3, padding=1),
            # ReLU activation
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        """ArithmeticError
        Inputs:
            x : Input tensor
        Outputs:
            Output tensor after passing through the DoubleConv block
        Performs the forward pass of the DoubleConv block.
        """
        # Passes the input tensor through the sequential convolutional layers and returns the output.
        return self.conv(x)

In [8]:
#################
## U-NET MODEL ##
#################
class UNet(nn.Module):
    """
    Creates a U-Net model for image segmentation tasks. The U-Net architecture consists of an encoder (downsampling path) and a decoder (upsampling path) with skip connections between corresponding layers. The purpose of this model is to perform pixel-wise classification for tasks such as medical image segmentation.
    """
    def __init__(self, n_classes):
        """
        Inputs:
            n_classes: Number of output classes for segmentation
        Outputs:
            None
        Initializes the U-Net model with the specified number of output classes.
        """
        super().__init__()
        # Encoder, consisting of four DoubleConv blocks with increasing channels. Its purpose is to extract features from the input image while reducing its spatial dimensions.
        self.d1 = DoubleConv(3, 64)
        self.d2 = DoubleConv(64, 128)
        self.d3 = DoubleConv(128, 256)
        self.d4 = DoubleConv(256, 512)
        
        # Bottleneck, the deepest part of the U-Net. It connects the encoder and decoder.
        self.bottleneck = DoubleConv(512, 1024)
        
        # Decoder, consisting of four upsampling layers followed by DoubleConv blocks. Its purpose is to reconstruct the spatial dimensions while combining features from the encoder via skip connections.
        self.u1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.c1 = DoubleConv(1024, 512)

        self.u2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.c2 = DoubleConv(512, 256)

        self.u3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.c3 = DoubleConv(256, 128)

        self.u4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.c4 = DoubleConv(128, 64)

        # Final output layer that maps to the desired number of classes for segmentation.
        self.out = nn.Conv2d(64, n_classes, 1)
        # Define a max pooling layer to be used in the encoder for downsampling the feature maps. This layer reduces the spatial dimensions by a factor of 2, allowing the network to capture more abstract features at deeper layers.
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        """
        Inputs:
            x : Input tensor
        Outputs:
            Output tensor after passing through the U-Net model
        Performs the forward pass of the U-Net model.
        """
        # Encoder, consisting of four DoubleConv blocks with increasing channels. Its purpose is to extract features from the input image while reducing its spatial dimensions.
        x1 = self.d1(x)
        x2 = self.d2(self.pool(x1))
        x3 = self.d3(self.pool(x2))
        x4 = self.d4(self.pool(x3))

        # Bottleneck, the deepest part of the U-Net. It connects the encoder and decoder.
        xb = self.bottleneck(self.pool(x4))

        # Decoder, consisting of four upsampling layers followed by DoubleConv blocks. Its purpose is to reconstruct the spatial dimensions while combining features from the encoder via skip connections.
        x = self.u1(xb)
        x = self.c1(torch.cat([x, x4], dim=1))

        x = self.u2(x)
        x = self.c2(torch.cat([x, x3], dim=1))

        x = self.u3(x)
        x = self.c3(torch.cat([x, x2], dim=1))

        x = self.u4(x)
        x = self.c4(torch.cat([x, x1], dim=1))

        return self.out(x)


In [9]:

############################
## IMPORT THE PET DATASET ##
############################
img_transform = T.Compose([
    T.Resize((256, 256), interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(),
    T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

# Transform function for the masks, For simplicity we will convert the
# current mask labels (1, 2, 3 for pet, background, and border) into binary 
# masks (pet vs background).
mask_transform = T.Compose([
    T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST),
    T.Lambda(lambda m: TF.pil_to_tensor(m).float()),  # ensures [1,H,W] for single-channel PIL
    T.Lambda(lambda m: (m > 0).float()),              # convert 1..3 → 1; 0 stays 0
])

# Load the Oxford-IIIT Pet dataset with the defined transforms. Here we just load the training/validation split.
full = OxfordIIITPet(root="./data", split="trainval",
                     target_types="segmentation", download=True,
                     transform=img_transform, target_transform=mask_transform)

# Reduce dataset size for faster experimentation.
N = 1500
subset_indices = list(range(N))
full = Subset(full, subset_indices)

# Split the dataset into training and validation sets (90% train, 10% val).
val_len = max(1, int(0.1 * len(full)))
train_len = len(full) - val_len
train_ds, val_ds = random_split(full, [train_len, val_len])

# Load the test split of the Oxford-IIIT Pet dataset.
test_ds = OxfordIIITPet(root="./data", split="test",
                        target_types="segmentation", download=True,
                        transform=img_transform, target_transform=mask_transform)

# Create DataLoaders for training, validation, and test sets.
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)


In [10]:
###################################
## DEFINE MODEL, LOSS, OPTIMIZER ##
###################################
# Define the U-Net model, loss function, and optimizer for training the model. BCEWithLogitsLoss is used for binary segmentation tasks, combining a sigmoid layer and binary cross-entropy loss in one single class. Adam is used as the optimizer for training. n_classes=1 for binary segmentation (pet vs background).
model = UNet(n_classes=1).to(device)
loss_func = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

#########################
## TRAIN/EVAL FUNCTION ##
#########################
def run_epoch(loader, train=True):
    """
    Inputs:
        loader: DataLoader for the dataset
        train : Boolean indicating whether to train or evaluate
    Outputs:
        loss_avg: Average loss over the epoch
    Create a function to run one epoch of training or evaluation.
    """
    # If the mode is training, set the model to training mode; otherwise, set it to evaluation mode.
    model.train() if train else model.eval()
    
    # Initialize counters for total samples and cumulative loss.
    total = 0
    loss_sum =  0.0

    # Iterate over the data in the DataLoader, one batch at a time. Set gradient tracking based on the mode (training or evaluation).
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            # If in training mode, zero the gradients of the optimizer.
            if train: optimizer.zero_grad(set_to_none=True)
            # Forward pass through the model to get logits (outputs)
            logits = model(x)
            # Compute the loss between the logits and the true labels.
            loss = loss_func(logits, y)
            # Backpropagation and optimization step if in training mode.
            if train:
                loss.backward()
                optimizer.step()
            # Update the cumulative loss.
            loss_sum += loss.item() * x.size(0)
            total += x.size(0)
    # Return the average loss for the epoch.
    return loss_sum / total


In [ ]:
#####################
## TRAIN THE MODEL ##
#####################
# Train the model for 1 epoch.
for epoch in range(1, 2):
    print("Epoch:", epoch)
    # Train and evaluate for one epoch. We use the training data to train
    # the model and the validation data to evaluate its performance.
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    

####################
## TEST THE MODEL ##
##################
test_loss = run_epoch(test_loader, False)
print("Test Loss:", test_loss)

# This takes about 35 minutes on my Mac for one epoch.

Epoch: 1
Test Loss: 0.0


In [13]:
va

0.0